In [ ]:
from nn import *
m = 256
def Cv(i=m, h=m, r=(7,7)):
    return Res(
        RMSNorm(i, elementwise_affine=0),
        Linear(i, h, 0),
        T(Conv1d(h, h, sum(r)+1, padding=max(r), groups=h, bias=0)),
        F(lambda _:_[...,max(r[1]-r[0],0):min(r[1]-r[0],0)or None,:]if r[1]!=r[0]else _),
        SiLU(),
        Linear(h, i, 0),
    )
def At(i=m, h=m//16, H=m//16, g=16, G=1, r=(7,7)):
    return Res(
        RMSNorm(i, elementwise_affine=0),
        Attn(i, h, H, g, G, r),
        Linear(G*g*(H or i), i, 0),
    )
def Ff(i=m, h=m):
    return Res(
        RMSNorm(i, elementwise_affine=0),
        SwiGLU(i, h),
        Linear(h, i, 0),
    )
def U(*_, i=m, h=m, s=2, S=0):
    return Res(
        Linear(i, h, 0),
        T(MaxPool1d(s)),
        *_,
        T(ConvTranspose1d(h, h, s, s, output_padding=max(S,0), groups=h, bias=0)),
        F(lambda _:_[...,S:,:]if S>0 else functional.pad(_[...,:S,:],(0,0,-S,0))if S else _),
        Linear(h, i, 0),
    )
m = Sequential(
    Embedding(256, m),
    At(),Ff(),
    U(At(),Ff(),
      U(At(),Ff(),
        U(At(),Ff(),
          U(At(),Ff(),
            U(At(),Ff(),
              U(At(),Ff(),
                U(At(),Ff()),
                At(),Ff()),
              At(),Ff()),
            At(),Ff()),
          At(),Ff()),
        At(),Ff()),
      At(),Ff()),
    At(),Ff(),
    Linear(m, 256),
).to('cuda',torch.bfloat16)
m = torch.compile(m, fullgraph=True, dynamic=False)
o = torch.optim.Adam(m.parameters(), 1/1024)
a = torch.from_numpy(np.memmap('py.256', mode='r'))
a = a.as_strided_((len(a)//4096-1,8192),(4096,1))
plot = Plot(2, 10, 10)
for y in range(1000000):
    t = a[torch.randint(len(a),[16])].to('cuda',torch.long)
    x = t.put(torch.randint(t.numel(),[t.numel()*3//4],device='cuda'),torch.randint(256,[t.numel()*3//4],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    (l*1024).backward()
    o.step()
    o.zero_grad()
    plot(l, lambda:(y.argmax(-1)==t).sum()/t.numel())